# The Similarity — Theory Playground

**Core thesis:** Similar dynamical processes can replay across time, assets, and timeframes. The broader the data bank, the more interesting the analogs become.

This notebook is your interactive research workstation. Eight sections, each building on the last:

1. **Data Universe** — what data do we have?
2. **Interactive Search** — run the 9-method pipeline with widget controls
3. **Match Deep-Dive** — inspect every match, every score, radar charts
4. **Forecast Cone** — percentile bands + Koopman dynamical forecast
5. **Cross-Asset Comparison** — does the pattern replay across assets?
6. **Backtest Validation** — walk-forward proof with calibration curves
7. **Method Ablation** — which of the 9 methods actually matter?
8. **Regime Analysis** — how does market regime affect match quality?

## Setup

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

NOTEBOOK_CWD = Path.cwd().resolve()
PLAYGROUND_ROOT = next(
    (path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "playground.py").exists()),
    None,
)
if PLAYGROUND_ROOT is None:
    candidate = NOTEBOOK_CWD / "the-similarity-playground"
    if (candidate / "playground.py").exists():
        PLAYGROUND_ROOT = candidate
if PLAYGROUND_ROOT is None:
    raise ModuleNotFoundError("Could not locate playground.py")
if str(PLAYGROUND_ROOT) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_ROOT))

from playground import (
    SCORE_FIELDS, SCORE_LABELS,
    bootstrap_imports, load_manifest, read_candles,
    run_local_search, run_backtest_local, run_ablation,
    top_matches_frame, forecast_summary_frame, theory_scorecard,
    plot_data_universe, plot_query_vs_matches, plot_score_radar,
    plot_forecast_cone, plot_terminal_distribution,
    plot_calibration_curve, plot_backtest_summary,
    plot_rolling_hit_rate, plot_p50_error_distribution,
    plot_ablation_results, plot_regime_price_chart,
    plot_hurst_rolling, plot_cross_asset_comparison,
    make_api_payload_from_dataset, run_api_search,
)

bootstrap_imports()
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
print("Ready.")

---
## §1 — Data Universe Explorer

What data do we have? How far back does each dataset go? How many bars?

In [ ]:
manifest = load_manifest()
display(manifest)

In [ ]:
fig = plot_data_universe(manifest)
fig.show()

---
## §2 — Interactive Search Workstation

Use the controls below to configure and run a similarity search. All 9 methods are available.

In [ ]:
# Build dropdown options from manifest
_manifest = load_manifest()
_asset_classes = sorted(_manifest["asset_class"].unique())
_symbols_by_ac = {ac: sorted(_manifest[_manifest["asset_class"] == ac]["symbol"].unique()) for ac in _asset_classes}
_tf_by_ac_sym = {}
for _, row in _manifest.iterrows():
    key = (row["asset_class"], row["symbol"])
    _tf_by_ac_sym.setdefault(key, []).append(row["timeframe"])

# Widgets
w_asset_class = widgets.Dropdown(options=_asset_classes, value="crypto", description="Asset Class")
w_symbol = widgets.Dropdown(options=_symbols_by_ac.get("crypto", []), description="Symbol")
w_timeframe = widgets.Dropdown(description="Timeframe")
w_window = widgets.IntSlider(value=120, min=20, max=500, step=10, description="Window Size")
w_topk = widgets.IntSlider(value=10, min=3, max=30, step=1, description="Top K")
w_stride = widgets.IntSlider(value=5, min=1, max=20, step=1, description="Stride")
w_forward = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Forward Bars")

# Method checkboxes
w_methods = {f: widgets.Checkbox(value=True, description=SCORE_LABELS.get(f, f), indent=False) for f in SCORE_FIELDS}
methods_box = widgets.VBox([widgets.Label("Active Methods:")] + list(w_methods.values()))

w_run = widgets.Button(description="Run Search", button_style="primary", icon="search")
w_output = widgets.Output()

# Dynamic cascading dropdowns
def _update_symbols(*_):
    ac = w_asset_class.value
    w_symbol.options = _symbols_by_ac.get(ac, [])
    if w_symbol.options:
        w_symbol.value = w_symbol.options[0]

def _update_timeframes(*_):
    key = (w_asset_class.value, w_symbol.value)
    tfs = sorted(set(_tf_by_ac_sym.get(key, [])))
    w_timeframe.options = tfs
    if tfs:
        w_timeframe.value = tfs[0]

w_asset_class.observe(_update_symbols, names="value")
w_symbol.observe(_update_timeframes, names="value")
_update_symbols()
_update_timeframes()

# State storage for downstream sections
search_state = {}

def _on_run(btn):
    with w_output:
        clear_output(wait=True)
        active = [f for f in SCORE_FIELDS if w_methods[f].value]
        if not active:
            print("Select at least one method.")
            return

        btn.disabled = True
        btn.description = "Running..."
        try:
            t0 = time.time()
            run = run_local_search(
                w_asset_class.value, w_symbol.value, w_timeframe.value,
                window_size=w_window.value, top_k=w_topk.value,
                stride=w_stride.value, forward_bars=w_forward.value,
                active_methods=active,
            )
            elapsed = time.time() - t0

            search_state["run"] = run
            search_state["params"] = {
                "asset_class": w_asset_class.value,
                "symbol": w_symbol.value,
                "timeframe": w_timeframe.value,
                "active_methods": active,
            }

            results = run["results"]
            forecast = run["forecast"]
            scorecard = theory_scorecard(results, forecast)

            print(f"Search completed in {elapsed:.1f}s — {len(results.matches)} matches found")
            print()
            for k, v in scorecard.items():
                print(f"  {k}: {v:.4f}")
        except Exception as e:
            print(f"Error: {e}")
        finally:
            btn.disabled = False
            btn.description = "Run Search"

w_run.on_click(_on_run)

controls = widgets.VBox([
    widgets.HBox([w_asset_class, w_symbol, w_timeframe]),
    widgets.HBox([w_window, w_topk, w_stride, w_forward]),
    methods_box,
    w_run,
])
display(controls, w_output)

---
## §3 — Match Deep-Dive

Run the search above, then execute this section to inspect matches in detail.

In [ ]:
# Overlay chart: query vs top matches
if "run" not in search_state:
    print("Run a search in §2 first.")
else:
    run = search_state["run"]
    fig = plot_query_vs_matches(run["query_frame"], run["results"], top_n=5)
    fig.show()

In [ ]:
# Full score breakdown table
if "run" in search_state:
    df = top_matches_frame(search_state["run"]["results"])
    display(df)

In [ ]:
# Radar chart for best match
if "run" in search_state:
    results = search_state["run"]["results"]
    if results.best:
        params = search_state.get("params", {})
        label = f"{params.get('symbol', '?')}/{params.get('timeframe', '?')}"
        fig = plot_score_radar(results.best, title=f"Best Match — {label} — Score Radar")
        fig.show()

In [ ]:
# Match navigator: inspect any match
if "run" in search_state:
    results = search_state["run"]["results"]
    match_selector = widgets.Dropdown(
        options=[(f"#{i+1} (score={m.confidence_score:.2f})", i) for i, m in enumerate(results.matches)],
        description="Match:",
    )
    nav_output = widgets.Output()

    def _show_match(change):
        with nav_output:
            clear_output(wait=True)
            match = results.matches[change["new"]]
            fig = plot_score_radar(match, title=f"Match #{change['new']+1} — Score Breakdown")
            fig.show()
            print(f"Period: {match.start_date} → {match.end_date}")
            print(f"Confidence: {match.confidence_score:.2f}")
            print(f"Regime: {match.regime or 'N/A'}")
            print(f"Transform R²: {match.transform_r2:.4f}")

    match_selector.observe(_show_match, names="value")
    display(match_selector, nav_output)
    # Trigger initial display
    _show_match({"new": 0})

---
## §4 — Forecast Cone

Percentile fan chart showing uncertainty bands (P10–P90), individual match forward paths, and Koopman dynamical forecast overlay.

In [ ]:
if "run" not in search_state:
    print("Run a search in §2 first.")
else:
    forecast = search_state["run"]["forecast"]

    fig = plot_forecast_cone(forecast, show_koopman=True)
    fig.show()

In [ ]:
# Terminal return distribution
if "run" in search_state:
    fig = plot_terminal_distribution(search_state["run"]["forecast"])
    fig.show()

In [ ]:
# Forecast summary table
if "run" in search_state:
    df = forecast_summary_frame(search_state["run"]["forecast"])
    display(df)

---
## §5 — Cross-Asset / Cross-Timeframe Comparison

Does the current query pattern replay across other assets and timeframes? Select datasets to compare.

In [ ]:
# Cross-asset comparison widget
_manifest_ca = load_manifest()
_all_datasets = [
    f"{r['asset_class']}/{r['symbol']}/{r['timeframe']}"
    for _, r in _manifest_ca.iterrows()
]

w_datasets = widgets.SelectMultiple(
    options=_all_datasets,
    value=_all_datasets[:3] if len(_all_datasets) >= 3 else _all_datasets,
    description="Datasets:",
    rows=10,
)
w_ca_run = widgets.Button(description="Compare", button_style="info", icon="chart-bar")
w_ca_output = widgets.Output()

cross_state = {}

def _on_compare(btn):
    with w_ca_output:
        clear_output(wait=True)
        selected = list(w_datasets.value)
        if len(selected) < 2:
            print("Select at least 2 datasets.")
            return

        btn.disabled = True
        btn.description = "Running..."
        scorecards = {}

        for ds in selected:
            parts = ds.split("/")
            ac, sym, tf = parts[0], parts[1], parts[2]
            try:
                run = run_local_search(
                    ac, sym, tf,
                    window_size=min(w_window.value, 100),
                    top_k=5, stride=10, forward_bars=w_forward.value,
                )
                sc = theory_scorecard(run["results"], run["forecast"])
                scorecards[ds] = sc
                print(f"  {ds}: confidence={sc['best_confidence']:.2f}, matches={sc['match_count']:.0f}")
            except Exception as e:
                print(f"  {ds}: SKIPPED — {e}")

        if len(scorecards) >= 2:
            cross_state["scorecards"] = scorecards
            fig = plot_cross_asset_comparison(scorecards)
            fig.show()

        btn.disabled = False
        btn.description = "Compare"

w_ca_run.on_click(_on_compare)
display(w_datasets, w_ca_run, w_ca_output)

---
## §6 — Backtest Validation

Walk-forward backtest: does the engine actually predict the future? No look-ahead, randomized trials, statistical rigor.

In [ ]:
w_bt_trials = widgets.IntSlider(value=20, min=5, max=100, step=5, description="Trials")
w_bt_run = widgets.Button(description="Run Backtest", button_style="warning", icon="flask")
w_bt_output = widgets.Output()

backtest_state = {}

def _on_backtest(btn):
    with w_bt_output:
        clear_output(wait=True)
        params = search_state.get("params")
        if not params:
            print("Run a search in §2 first to set the asset/timeframe.")
            return

        btn.disabled = True
        btn.description = "Running..."
        progress_label = widgets.Label(value="Starting...")
        display(progress_label)

        def _progress(completed, total):
            progress_label.value = f"Trial {completed}/{total}"

        try:
            t0 = time.time()
            report = run_backtest_local(
                params["asset_class"], params["symbol"], params["timeframe"],
                window_size=w_window.value,
                forward_bars=w_forward.value,
                n_trials=w_bt_trials.value,
                active_methods=params.get("active_methods"),
                progress_fn=_progress,
            )
            elapsed = time.time() - t0
            backtest_state["report"] = report
            progress_label.value = f"Done in {elapsed:.1f}s"

            print(report.summary())
        except Exception as e:
            print(f"Error: {e}")
        finally:
            btn.disabled = False
            btn.description = "Run Backtest"

w_bt_run.on_click(_on_backtest)
display(widgets.HBox([w_bt_trials, w_bt_run]), w_bt_output)

In [ ]:
# Backtest charts
if "report" not in backtest_state:
    print("Run the backtest above first.")
else:
    report = backtest_state["report"]

    fig = plot_backtest_summary(report)
    fig.show()

    fig = plot_calibration_curve(report)
    fig.show()

    fig = plot_rolling_hit_rate(report)
    fig.show()

    fig = plot_p50_error_distribution(report)
    fig.show()

---
## §7 — Method Ablation

Which of the 9 methods actually improve predictions? This runs N+1 backtests: one baseline with all methods, then one with each method removed. Red bars = removing that method *hurt* performance (the method was valuable).

**Note:** This takes ~10x longer than a single backtest. Use fewer trials for faster iteration.

In [ ]:
w_abl_trials = widgets.IntSlider(value=15, min=5, max=50, step=5, description="Trials/run")
w_abl_run = widgets.Button(description="Run Ablation", button_style="danger", icon="scissors")
w_abl_output = widgets.Output()

ablation_state = {}

def _on_ablation(btn):
    with w_abl_output:
        clear_output(wait=True)
        params = search_state.get("params")
        if not params:
            print("Run a search in §2 first.")
            return

        btn.disabled = True
        btn.description = "Running..."
        progress_label = widgets.Label(value="Starting...")
        display(progress_label)

        def _progress(method_name, step, total):
            progress_label.value = f"[{step+1}/{total}] {'Baseline' if method_name == 'baseline' else f'Without {SCORE_LABELS.get(method_name, method_name)}'}"

        try:
            t0 = time.time()
            df = run_ablation(
                params["asset_class"], params["symbol"], params["timeframe"],
                window_size=w_window.value,
                forward_bars=w_forward.value,
                n_trials=w_abl_trials.value,
                progress_fn=_progress,
            )
            elapsed = time.time() - t0
            ablation_state["df"] = df
            progress_label.value = f"Done in {elapsed:.1f}s"

            display(df)
        except Exception as e:
            print(f"Error: {e}")
        finally:
            btn.disabled = False
            btn.description = "Run Ablation"

w_abl_run.on_click(_on_ablation)
display(widgets.HBox([w_abl_trials, w_abl_run]), w_abl_output)

In [ ]:
# Ablation chart
if "df" not in ablation_state:
    print("Run the ablation above first.")
else:
    df = ablation_state["df"]
    # Exclude baseline row for the chart
    chart_df = df[df["removed_method"] != "(baseline)"].copy()
    fig = plot_ablation_results(chart_df)
    fig.show()

    # Recommendation
    helpful = chart_df[chart_df["delta_hit_rate"] < -0.01]["removed_method"].tolist()
    harmful = chart_df[chart_df["delta_hit_rate"] > 0.01]["removed_method"].tolist()

    print("
Method Analysis:")
    if helpful:
        print(f"  Valuable (removing hurts): {', '.join(SCORE_LABELS.get(m, m) for m in helpful)}")
    if harmful:
        print(f"  Questionable (removing helps): {', '.join(SCORE_LABELS.get(m, m) for m in harmful)}")
    if not helpful and not harmful:
        print("  All methods contribute roughly equally.")

---
## §8 — Regime Analysis

How does market regime affect the engine? Regime-colored price charts, rolling Hurst exponent, and regime transition analysis.

In [ ]:
if "run" not in search_state:
    print("Run a search in §2 first.")
else:
    history_frame = search_state["run"]["history_frame"]

    fig = plot_regime_price_chart(history_frame)
    fig.show()

In [ ]:
# Rolling Hurst exponent
if "run" in search_state:
    fig = plot_hurst_rolling(search_state["run"]["history_frame"])
    fig.show()

In [ ]:
# Regime distribution
if "run" in search_state:
    from the_similarity.core.regime import tag_regime

    history_frame = search_state["run"]["history_frame"]
    prices = history_frame["close"].values
    window = w_window.value

    regime_counts = {}
    for i in range(0, len(prices) - window + 1, 30):
        regime = tag_regime(prices[i:i + window])
        regime_counts[regime] = regime_counts.get(regime, 0) + 1

    print("Regime Distribution:")
    total = sum(regime_counts.values())
    for regime, count in sorted(regime_counts.items(), key=lambda x: -x[1]):
        print(f"  {regime:20s}: {count:4d} windows ({count/total:.1%})")

In [ ]:
# Match quality by regime
if "run" in search_state:
    results = search_state["run"]["results"]
    regime_scores = {}
    for match in results.matches:
        r = match.regime or "unknown"
        regime_scores.setdefault(r, []).append(match.confidence_score)

    print("Match Quality by Regime:")
    for regime, scores in sorted(regime_scores.items()):
        avg = np.mean(scores)
        print(f"  {regime:20s}: avg_confidence={avg:.2f}, n={len(scores)}")

---
## Bonus — API Comparison

Compare local engine results with the FastAPI backend (if running).

In [ ]:
if "run" in search_state:
    params = search_state["params"]
    payload = make_api_payload_from_dataset(
        params["asset_class"], params["symbol"], params["timeframe"],
        window_size=w_window.value, top_k=w_topk.value,
        stride=w_stride.value, forward_bars=w_forward.value,
        active_methods=params.get("active_methods"),
    )

    try:
        api_response = run_api_search(payload)
        print(f"API matches: {len(api_response['matches'])}")
        print(f"API forecast bars: {api_response['forecast']['bars']}")
        display(pd.DataFrame(api_response["matches"]).head())
    except Exception as e:
        print(f"API not available. Start with ./dev.sh backend")
        print(f"Error: {e}")